# Quantizing a Jet Tagger with Brevitas


In [ ]:
# Google Colab setup: uncomment and run this cell once to install every
# dependency used by this notebook, then restart the runtime if prompted.

# # Brevitas (tutorial branch) and the core scientific stack
# !pip install -q --upgrade pip wheel "setuptools<81" setuptools_scm
# !pip install -q git+https://github.com/Xilinx/brevitas.git@giuseppe/new_tutorial
# !pip install -q numpy scikit-learn matplotlib
#
# # Export tooling and the Netron graph viewer
# !pip install -q 'onnx<1.23' 'qonnx>=0.5.0' onnxoptimizer netron
#
# # Optional hls4ml downstream conversion and its model plotter
# !pip install -q hls4ml pydot
# !apt-get -qq install -y graphviz   # provides the `dot` executable that pydot calls


## Quantization with Brevitas: setting the scene

Brevitas is a PyTorch library for neural-network quantization. Quantized models remain PyTorch models, with simulated quantization expressed through quantized layers and quantizers. This lets us calibrate, fine-tune, inspect, and eventually export the quantized model.

Throughout the notebook, we will pay particular attention to what is quantized, how it is quantized, and where quantization boundaries are placed.

We will do this by following two typical quantization flows: in the first case, we'll use mostly out-of-the-box components available in Brevitas; in the second one, we'll see how to extend Brevitas with new custom-built components.


### Quick reference

This table is here as a quick reference during the hands-on exercises.

| Concept | In this notebook |
| --- | --- |
| **Quantizer** | Defines how a tensor is quantized: format or bit width, scale, rounding, granularity, and related choices. |
| **Quantized layer** | A PyTorch layer with quantization applied to tensor roles such as inputs, weights, biases, or outputs. |
| **RTN (Round-to-Nearest)** | Maps floating-point values to the nearest representable quantized value for a given scale and range. |
| **Calibration** | Uses representative data to determine activation quantization parameters. |
| **PTQ (Post-Training Quantization)** | Quantizes an already-trained floating-point model without full retraining. Calibration and methods such as Qronos are part of the PTQ workflow shown here. |
| **Qronos** | A PTQ method used to reduce quantization error and recover accuracy. |
| **QAT (Quantization-Aware Training)** | Fine-tunes the model while simulated quantization is active. |
| **Export** | Converts the Brevitas model into a representation such as QONNX for downstream use. |

## What we will do

**FP32 model -> quantized model -> calibration -> PTQ -> QAT -> export**

1. Start with a small floating-point jet-tagging network.
2. Replace its layers with quantized Brevitas layers.
3. Inspect what is actually being quantized.
4. Calibrate activation quantizers using representative data.
5. Apply PTQ and short QAT for accuracy recovery.
6. Export the resulting model to QONNX.
7. In the advanced section, build custom quantizers and learned integer formats.

## Setup and baseline

We begin by preparing a reproducible environment, loading and standardizing the jet-tagging data, and establishing the floating-point model that the quantized variants will be compared against.

### Reproducibility

The examples use fixed random seeds, a fixed dataset split, and reported software versions. This keeps differences between model variants attributable to the quantization changes as far as practical. CPU and GPU results may still differ slightly.

The input features are standardized with training-set statistics. They contain both positive and negative values, so the model input requires a signed quantizer.

### Environment and random state

Next we need to import the packages used throughout the notebook, select the available CPU (central processing unit) or GPU (graphics processing unit), and record the software versions used for this run.

In [ ]:
from pathlib import Path
import math
import platform
import random

import brevitas
import brevitas.nn as qnn
from importlib.metadata import version
import numpy as np
import sklearn
import torch
from sklearn.datasets import fetch_openml
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
import warnings

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ARTIFACT_DIR = Path('tutorial_artifacts')
ARTIFACT_DIR.mkdir(exist_ok=True)

print({
    'python': platform.python_version(),
    'torch': torch.__version__,
    'brevitas': brevitas.__version__,
    'scikit-learn': sklearn.__version__,
    'onnx': version('onnx'),
    'qonnx': version('qonnx'),
    'device': str(DEVICE),
})

### Jet-tagging data

First we prepare the data.

Load the OpenML dataset, create a stratified subset, and standardize it with training-set statistics. The same split is used by every model variant.

In [ ]:
# OpenML ID 42468: 16 high-level LHC jet features and five jet classes.
dataset = fetch_openml(data_id=42468, as_frame=False)
X = np.asarray(dataset.data, dtype=np.float32)
labels = LabelEncoder().fit_transform(dataset.target)

X_train, X_test, y_train, y_test = train_test_split(
    X, labels, train_size=40_000, test_size=10_000, random_state=SEED, stratify=labels)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype(np.float32)
X_test = scaler.transform(X_test).astype(np.float32)

train_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train).long()),
    batch_size=1024, shuffle=True, generator=torch.Generator().manual_seed(SEED))
calibration_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_train[:8192]), torch.from_numpy(y_train[:8192]).long()),
    batch_size=1024, shuffle=False)

X_train_t = torch.from_numpy(X_train)
X_test_t = torch.from_numpy(X_test)
y_test_t = torch.from_numpy(y_test).long()
print(f'train={X_train.shape}, test={X_test.shape}, classes={np.unique(labels)}')

### Floating-point baseline

All quantized variants use the same preprocessing and start from these floating-point weights.

The checkpoint is cached in the notebook runtime.

First we define some utility functions that we will re-use throughout the notebook, to train and evaluate the model (either float or quantized):

In [ ]:
def evaluate(model):
    model.eval()
    predictions = []
    with torch.no_grad():
        for start in range(0, len(X_test_t), 2048):
            logits = model(X_test_t[start:start + 2048].to(DEVICE))
            predictions.append(logits.argmax(dim=1).cpu())
    return accuracy_score(y_test_t.numpy(), torch.cat(predictions).numpy())


def train_steps(model, loader, optimizer, max_steps, regularizer=None, post_step=None):
    model.train()
    criterion = nn.CrossEntropyLoss()
    losses = []
    for step, (features, targets) in enumerate(loader):
        if step >= max_steps:
            break
        features, targets = features.to(DEVICE), targets.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(features), targets)
        if regularizer is not None:
            loss = loss + regularizer(model)
        loss.backward()
        optimizer.step()
        if post_step is not None:
            post_step(model)
        losses.append(loss.detach().item())
    return losses

Afterwards we need to define the model structure, load the checkpoint (or pre-load one if available).\
Notice how the model is a mix of modular (nn.Linear) and functional (torch.relu) calls.

In [ ]:
class JetTagger(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(16, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 32)
        self.output = nn.Linear(32, 5)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.relu(self.fc3(x))
        return self.output(x)


fp32_path = ARTIFACT_DIR / 'jet_tagger_fp32.pt'
fp32_model = JetTagger().to(DEVICE)
if fp32_path.exists():
    fp32_model.load_state_dict(torch.load(fp32_path, map_location=DEVICE, weights_only=True))
else:
    optimizer = torch.optim.Adam(fp32_model.parameters(), lr=3e-3)
    for _ in range(3):
        train_steps(fp32_model, train_loader, optimizer, max_steps=len(train_loader))
    torch.save(fp32_model.state_dict(), fp32_path)

fp32_accuracy = evaluate(fp32_model)
print(f'FP32 accuracy: {fp32_accuracy:.3f}; parameters: {sum(p.numel() for p in fp32_model.parameters()):,}')

fp32_state = {
    name: value.detach().cpu().clone()
    for name, value in fp32_model.state_dict().items()}
example_input_cpu = torch.from_numpy(X_test[:1]).contiguous()

### Looking at the network

Before quantizing anything, we draw the floating-point model so we can compare the diagrams of the later variants against it. The renderer walks the module tree and reports each layer's role and numerical format without exporting a graph.

In [ ]:
# The workshop ships a small ASCII network renderer next to this notebook.
# On Colab the notebook is opened on its own, so fetch the helper if missing.
from pathlib import Path

if not Path('notebook_utils.py').exists():
    import urllib.request
    url = (
        'https://raw.githubusercontent.com/Xilinx/brevitas/'
        'giuseppe/new_tutorial/notebooks/notebook_utils.py')
    urllib.request.urlretrieve(url, 'notebook_utils.py')

from notebook_utils import draw_network

draw_network(fp32_model, title='FP32 jet tagger', orientation='horizontal')

## How quantization is represented in Brevitas

Before quantizing the full model, let's look at how the concepts from the introduction actually look in code. We will examine three of the main building blocks used throughout the tutorial: quantizer recipes, quantized layers, and `QuantTensor`s.

To create a reusable quantizer recipe, we subclass the closest predefined quantizer and override only the policy attributes required by the experiment.

Quantizers in Brevitas are resolved through dependency injection.\
Intuitively, they can be considered as flattened namespaces: the dependency injector engine will automatically wire the attributes to the corresponding class.

In [ ]:
from brevitas.core.scaling import ScalingImplType
from brevitas.quant import Int32Bias
from brevitas.quant import Int8ActPerTensorFloat
from brevitas.quant import Int8WeightPerChannelFloat


class Int4WeightPerChannelFloat(Int8WeightPerChannelFloat):
    bit_width = 4


for attribute in (
        'bit_width', 'scaling_impl_type', 'scaling_stats_op',
        'scaling_per_output_type', 'float_to_int_impl_type'):
    print(f'{attribute}: {getattr(Int4WeightPerChannelFloat, attribute)}')

### Quantized layers

A quantized layer assigns quantizer recipes to specific tensor roles. A `QuantLinear` owns separate proxies for its input, weight, bias, and output; each proxy resolves the corresponding quantizer recipe.

In [ ]:
recap_linear = qnn.QuantLinear(
    4,
    3,
    input_quant=Int8ActPerTensorFloat,
    weight_quant=Int4WeightPerChannelFloat,
    bias=True,
    bias_quant=Int32Bias,
    return_quant_tensor=True)

for role in ('input_quant', 'weight_quant', 'bias_quant', 'output_quant'):
    proxy = getattr(recap_linear, role)
    if hasattr(proxy, 'fused_activation_quant_proxy'):
        tensor_quant = proxy.fused_activation_quant_proxy
    elif hasattr(proxy, 'tensor_quant'):
        tensor_quant = proxy.tensor_quant
    else:
        tensor_quant = None
    print(f'{role}: proxy={type(proxy).__name__}, tensor_quant={type(tensor_quant).__name__}')

### QuantTensors

A `QuantTensor` is the object exchanged between compatible quantized layers. It carries the fake-quantized value together with scale, zero-point, precision, and signedness.

In the next cell we reconstruct the represented value and check that the same object reaches a second quantized layer.

In [ ]:
from brevitas.quant_tensor import QuantTensor

recap_identity = qnn.QuantIdentity(
    act_quant=Int8ActPerTensorFloat, return_quant_tensor=True)
recap_input = torch.randn(2, 3)
recap_output = recap_identity(recap_input)

print(type(recap_output).__name__)
print('value shape:', tuple(recap_output.value.shape))
print('integer sample:', recap_output.int()[0].cpu())
print('scale:', recap_output.scale.detach().cpu().flatten())
print('zero point:', recap_output.zero_point.detach().cpu().flatten())
print('bit width:', recap_output.bit_width.detach().cpu().flatten())
print('signed:', recap_output.signed)

reconstructed_output = (
    recap_output.int().type_as(recap_output.value) - recap_output.zero_point
) * recap_output.scale
print('maximum reconstruction error:',
      (reconstructed_output - recap_output.value).abs().max().item())

recap_next = qnn.QuantLinear(
    3, 2, weight_quant=Int4WeightPerChannelFloat, bias=False)
captured_input = {}

def capture_recap_input(module, args):
    captured_input['value'] = args[0]

hook = recap_next.register_forward_pre_hook(capture_recap_input)
recap_next(recap_output)
hook.remove()
print('next layer receives QuantTensor:',
      isinstance(captured_input['value'], QuantTensor))

### How to apply quantization in Brevitas

Brevitas offers multiple ways to convert a floating-point model into a quantized version.

Each approach has its own tradeoffs. We recommend choosing the simplest one that meets the requirements for:
- performance
- accuracy
- compatibility with the export flow and compiler toolchain

The following table summarizes the four currently supported approaches.


| Approach | What it changes | Main advantage | Main limitation |
| --- | --- | --- | --- |
| Layerwise quantization | Replaces supported module instances | Simple mapping and automatic state-dict preservation | Functional calls are not replaced |
| FX-based quantization | Rewrites modules and operations visible in a traced graph | Can place quantization boundaries at graph level | Requires a traceable model |
| Functional quantization (WIP) | Intercepts selected PyTorch operations at runtime | Can reach functional operators without redefining the model | More difficult to configure and not yet suited to every use case |
| Full-model replacement | Defines the quantized model explicitly | Full control over tensor roles and metadata propagation | Requires maintaining a compatible model hierarchy |



## Basic quantization workflow with Brevitas

The first part of the tutorial follows the most common workflow: replace supported modules, use predefined or lightly customized quantizers, recover accuracy with PTQ and QAT, and export through a built-in path. This approach is appropriate when the model does not require quantizing unsupported functional calls or complex residual pathways.


## Integer quantizer recipes

Before replacing any layers, we define the integer recipes used by the conversion map. 

In Brevitas some configurations can be changed using special enums or strings. These attributes will define what classes and configurations should be used when building the quantizers.

Examples of this are:
- Stats to use for scale factors (MAX, MEAN, PERCENTILE, ...)
- Stats to use for zero point (MIN, PERCENTILE, ...)
- Scale/zero-point implementation type (STATS, CONST, PARAMETER_FROM_STATS, ...)
- Scale/zero-point granularity (TENSOR, CHANNEL, GROUP)

This avoids having to pass class types or know which classes to change to switch from one setting to another.

The weight recipes initialize one scale per weight tensor from maximum statistics. The activation recipes initialize one scale per tensor from the 99th percentile of representative runtime values.

In [ ]:
from brevitas.core.scaling import ScalingImplType
from brevitas.inject.enum import StatsOp
from brevitas.quant import Int32Bias
from brevitas.quant import Int8ActPerTensorFloat
from brevitas.quant import Int8WeightPerTensorFloat

# Weight Quantization
class Int8WeightPerTensorParameterFloat(Int8WeightPerTensorFloat):
    scaling_impl_type = ScalingImplType.PARAMETER_FROM_STATS
    scaling_stats_op = StatsOp.MAX

class Int3WeightPerTensorParameterFloat(Int8WeightPerTensorFloat):
    scaling_impl_type = ScalingImplType.PARAMETER_FROM_STATS
    scaling_stats_op = StatsOp.MAX
    bit_width = 3


# Input quantization
class Int8ActPerTensorFloatNewPercentile(Int8ActPerTensorFloat):
    scaling_stats_op = StatsOp.PERCENTILE
    high_percentile_q = 99.0

class Int4ActPerTensorFloatNewPercentile(Int8ActPerTensorFloat):
    scaling_stats_op = StatsOp.PERCENTILE
    high_percentile_q = 99.0
    bit_width = 4


## Layerwise quantization


`layerwise_quantize` represents one of the simplest ways to insert quantization nodes in your model.

All it needs is a mapping between floating point layer types and their quantized counterparts.

It will automatically transfer all the matching attributes (e.g., input or output features, bias, etc.), as well as preserve the state dict.

The user has to specify the quantized arguments, either directly or using lambda functions.

In [ ]:
from brevitas.graph.quantize import layerwise_quantize

linear_compute_layer_map = {
    nn.Linear: (
        qnn.QuantLinear,
        {
            'input_quant': lambda module: Int8ActPerTensorFloatNewPercentile if module.out_features == 5 else Int4ActPerTensorFloatNewPercentile,
            'weight_quant': lambda module: Int8WeightPerTensorParameterFloat if module.out_features == 5 else Int3WeightPerTensorParameterFloat,
            'bias_quant': Int32Bias,
            'return_quant_tensor': False,
        },
    ),
}
replacement_class, linear_quantizers = linear_compute_layer_map[nn.Linear]
print('Default nn.Linear replacement:', replacement_class.__name__)
for role in ('input_quant', 'weight_quant', 'bias_quant'):
    print(f'  {role}: {linear_quantizers[role].__name__}')
print('  return_quant_tensor:', linear_quantizers['return_quant_tensor'])

Once the layerwise replacement has been applied, we can inspect the new model.

In particular, we check that the hidden layers use three-bit weights, the output layer uses eight-bit weights, and the original floating-point parameters are preserved. The compact diagram below summarizes the converted layers and their weight and activation formats.

In [ ]:
model = JetTagger().to(DEVICE)
model.load_state_dict(fp32_state)
quant_model = layerwise_quantize(
    model, compute_layer_map=linear_compute_layer_map).to(DEVICE)



In [ ]:
draw_network(quant_model, title='Layerwise integer jet tagger', mode='compact', orientation='horizontal')

### Takeaway

The layerwise map has preserved the dense parameters and applied the requested weight precisions.

The functional `torch.relu` calls are still executed in floating point, and their outputs are quantized when they enter the following `QuantLinear`. We now have quantized layers and explicit boundaries, but the activation quantizers still need representative data to determine suitable scales.

## Calibration and post-training quantization (PTQ)

We have now replaced the floating-point layers with quantized layers. Weight quantization parameters can be derived directly from the model weights, but activation quantization parameters need to be calibrated using representative data. Before calibration, we first inspect the effect of weight quantization and the initial activation quantizers.

A quantizer needs parameters such as scale and, for asymmetric schemes, zero-point. These values can be fixed, computed from statistics, initialized from statistics and stored as parameters, or learned directly. In this example, the weight quantizers initialize one scale per weight tensor from maximum weight statistics and then store those values as parameters. Brevitas also provides other statistics operations and supports custom ones.



In [ ]:
probe_scaling = quant_model.fc1.weight_quant.tensor_quant.scaling_impl.init_done
print('init_done before quant_weight():', bool(probe_scaling))

quant_model.fc1.quant_weight()
probe_scaling = quant_model.fc1.weight_quant.tensor_quant.scaling_impl.init_done
print('init_done after quant_weight():', bool(probe_scaling))

weight_qt = quant_model.fc1.quant_weight()
reconstructed = (weight_qt.int() - weight_qt.zero_point) * weight_qt.scale
print('weight scale shape:', tuple(weight_qt.scale.shape))
print('weight bit width:', weight_qt.bit_width.item())
print('max FP32-to-quantized weight error:', (quant_model.fc1.weight - weight_qt.value).abs().max().item())
print('accuracy before explicit calibration:', evaluate(quant_model))

### Activation calibration

Activations do not have an associated persistent tensor. Their statistics have to be collected from representative inputs. The activation quantizers used here convert those runtime statistics into scale parameters during calibration.

One possibility to define activation scales is to use `calibration_mode`.
The idea is to track statistics across a set of representative data, while we keep quantization disabled, and then set the scale appropriately.

There is no loss, backward pass, or optimizer step. The state snapshots check that quantizer state changes while dense weights and biases remain unchanged.

The newly initialized scale, and potentially the zero-point, subsequently become parameters that can still be adjusted by gradient descent.

In [ ]:
from brevitas.graph.calibrate import calibration_mode

# First we observe the scale before calibration
calibration_sample = next(iter(calibration_loader))[0]
calibration_sample = calibration_sample.to(DEVICE)
quant_model.eval()


# We collect output and scale before calibration
with torch.no_grad():
    output_before_calibration = quant_model(calibration_sample).detach().cpu()
    input_scale_before = quant_model.fc1.input_quant(calibration_sample).scale.detach().cpu().clone()
state_before_calibration = {
    name: value.detach().cpu().clone() for name, value in quant_model.state_dict().items()}
parameters_before_calibration = {
    name: value.detach().cpu().clone() for name, value in quant_model.named_parameters()}


In [ ]:
# This is all it takes to calibrate your model:
# enter the context manager and call the forward pass for as many samples as you need
with torch.no_grad(), calibration_mode(quant_model):
    for features, _ in calibration_loader:
        quant_model(features.to(DEVICE))

Calibration has now collected the activation statistics. We can compare the saved state and verify that quantizer state changed while the dense parameters remained unchanged.

In [ ]:
state_after_calibration = quant_model.state_dict()
changed_state = [
    name for name, value in state_after_calibration.items()
    if name not in state_before_calibration
    or not torch.equal(state_before_calibration[name], value.detach().cpu())]
changed_parameters = [
    name for name, value in quant_model.named_parameters()
    if not torch.equal(parameters_before_calibration[name], value.detach().cpu())]

The state comparison has separated quantizer changes from dense parameters. We can now compare the input scale and model output before and after calibration, and evaluate the calibrated model.

In [ ]:
with torch.no_grad():
    output_after_calibration = quant_model(calibration_sample).detach().cpu()
    input_scale_after = quant_model.fc1.input_quant(calibration_sample).scale.detach().cpu().clone()
torch.save(
    quant_model.state_dict(), ARTIFACT_DIR / 'jet_tagger_integer_calibrated.pt')
print('state entries changed by calibration:')
for name in changed_state:
    print(' ', name)
# print('dense parameters changed:', changed_dense_parameters)
# print('quantizer parameters initialized from statistics:', changed_quantizer_parameters)
print('input scale before calibration:', input_scale_before.flatten())
print('input scale after calibration:', input_scale_after.flatten())
print('maximum output change:', (output_after_calibration - output_before_calibration).abs().max().item())


Finally we check the accuracy of the calibrated model:

In [ ]:
ptq_accuracy = evaluate(quant_model)
print(f'calibrated PTQ accuracy: {ptq_accuracy:.3f}')

### Takeaway

Calibration uses unlabeled representative data to initialize activation quantization parameters; it does not update the model's dense weights or biases. The calibration set must still cover the activation ranges expected at inference, because its representativeness directly affects quantized accuracy.

The result is a calibrated PTQ model. Next, we use Qronos to further reduce quantization error before moving on to QAT.


### Improving PTQ accuracy with Qronos

Although Brevitas was created with QAT in mind, it has now first-class support for PTQ accuracy-recovery algorithms.

Qronos builds on greedy layer-wise reconstruction methods such as GPTQ to reduce quantization error and recover accuracy.

Before continuing, we define two small helpers for the parameter snapshots used by Qronos and QAT. The calibration loops remain explicit in both workflows.

In [ ]:
from notebook_utils import snapshot_parameters
from notebook_utils import changed_parameter_names

The calibrated fixed-format `quant_model` can now be refined with Qronos [5]. We save the Qronos checkpoint before continuing QAT on the same model.

Qronos can improve accuracy by correcting quantization error one layer at a time while accounting for error propagated from previous layers.

Because of this, it requires N passes over the calibration set, where N is the total number of quantized layers in your network.

In [ ]:
from functools import partial
from brevitas.graph.gpfq import gpfq_mode
from brevitas.graph.qronos import Qronos
from brevitas.graph.utils import remove_weight_orig

qronos_before = snapshot_parameters(quant_model)
qronos_accuracy_before = evaluate(quant_model)

with torch.no_grad(), gpfq_mode(
        model=quant_model,
        algorithm_impl=Qronos,
        use_quant_activations=False,
        act_order=True,
        device='same') as qronos:
    for _ in range(qronos.num_layers):
        for features, _ in calibration_loader:
            qronos.model(features.to(DEVICE))
        qronos.update()

qronos_changed = changed_parameter_names(qronos_before, quant_model)

print('Qronos changed parameters:', qronos_changed)
print('accuracy before Qronos:', qronos_accuracy_before)
print('accuracy after Qronos:', evaluate(quant_model))

# We try as hard as we can to make non-destructive changes.
# In algorithms like GPTQ and Qronos, this means storing one extra tensor
# with the original floating point weights.
# If we no longer need them, this helper removes them to save memory.
remove_weight_orig(quant_model)
torch.save(
    quant_model.state_dict(), ARTIFACT_DIR / 'jet_tagger_integer_qronos.pt')

As we can see, Qronos has updated the dense weights. We remove the temporary `weight_orig` buffers, save this PTQ stage, and continue with QAT on the same `quant_model`.

### QAT after Qronos

Brevitas does not impose a strict distinction between PTQ and QAT, so the same modules and quantizers can be used seamlessly for either accuracy-recovery approach.

This is immediately evident in the next phase; after applying Qronos, we can simply continue with QAT without any change to our model.

Notably, `quant_model.parameters()` contains both the weights of the model, as well as quantization's parameters like scale factors (and zero-point, in case of asymmetric quantization).

In [ ]:
parameter_names = [name for name, _ in quant_model.named_parameters()]
quantizer_parameter_names = [
    name for name in parameter_names if 'quant' in name]
print('total parameters:', len(parameter_names))
print('quantizer parameters optimized alongside the weights:',
      len(quantizer_parameter_names))
for name in quantizer_parameter_names[:3]:
    print(' ', name)

qat_optimizer = torch.optim.Adam(quant_model.parameters(), lr=1e-3)
qat_losses = train_steps(
    quant_model, train_loader, qat_optimizer, max_steps=20)

print('short-QAT accuracy:', evaluate(quant_model))
torch.save(
    quant_model.state_dict(), ARTIFACT_DIR / 'jet_tagger_integer_qat.pt')

Once the accuracy-recovery phase is finished, we are ready to export our model.

### Built-in QONNX export

Brevitas offers multiple export paths, including but not limited to:
- ONNX
- QONNX
- TorchScript
- vLLM
- GGUF

For this example, we use the built-in QONNX exporter.

In [ ]:
import onnx
from brevitas.export import export_qonnx
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.core.onnx_exec import execute_onnx
from qonnx.transformation.infer_shapes import InferShapes

export_original_device = next(quant_model.parameters()).device
export_was_training = quant_model.training
quant_model.cpu().eval()
integer_qonnx_path = ARTIFACT_DIR / 'jet_tagger_integer_qonnx.onnx'
try:
    export_qonnx(
        quant_model,
        args=example_input_cpu,
        export_path=str(integer_qonnx_path),
        opset_version=14,
        dynamo=False)
    integer_onnx_model = onnx.load(integer_qonnx_path)
    onnx.checker.check_model(integer_onnx_model)
    integer_qonnx_model = ModelWrapper(integer_onnx_model).transform(InferShapes())

    initializer_names = {
        initializer.name for initializer in integer_onnx_model.graph.initializer}
    input_name = next(
        value.name for value in integer_onnx_model.graph.input
        if value.name not in initializer_names)
    output_name = integer_onnx_model.graph.output[0].name
    qonnx_output = execute_onnx(
        integer_qonnx_model, {input_name: example_input_cpu.numpy()})[output_name]
    with torch.no_grad():
        torch_output = quant_model(example_input_cpu).numpy()
    export_max_error = float(abs(torch_output - qonnx_output).max())
    export_tolerance = 1e-4
    print('maximum PyTorch/QONNX difference:', export_max_error)
    if export_max_error > export_tolerance:
        raise RuntimeError(
            f'Export parity exceeded {export_tolerance}: {export_max_error}')
finally:
    quant_model.to(export_original_device)
    quant_model.train(export_was_training)

The exported QONNX file now carries the quantization decisions as `Quant` nodes. We open it in Netron to see where they sit: the weight and bias quantizers feed each `Gemm`, and selecting a `Quant` node shows its scale, zero point, and bit width. If the viewer stays empty, open the `.onnx` file at [netron.app](https://netron.app).

In [ ]:
from notebook_utils import show_netron

show_netron(integer_qonnx_path)

### hls4ml conversion

This cell converts the fixed-integer QONNX artifact into an hls4ml project. It does not run synthesis and is excluded from the timed path.

hls4ml reads the quantization directly from the `Quant` nodes, so we only clean the graph and convert its `Gemm` nodes to `MatMul`.

This is the reason the weight recipes use one scale per tensor. hls4ml can keep the integer weights and apply the scale in a separate multiplication after the layer. In hls4ml 1.3.0 a per-channel weight scale cannot be moved past the matrix multiplication, and the conversion fails on it.

In [ ]:
try:
    import hls4ml
except ModuleNotFoundError as error:
    raise RuntimeError(
        'Install the tutorial-tested hls4ml version before running this optional cell') from error
import qonnx.util.cleanup
from qonnx.transformation.gemm_to_matmul import GemmToMatMul

hls_qonnx_path = ARTIFACT_DIR / 'jet_tagger_integer_hls4ml.onnx'
qonnx.util.cleanup.cleanup(str(integer_qonnx_path), out_file=str(hls_qonnx_path))
hls_source_model = ModelWrapper(str(hls_qonnx_path))
hls_source_model = hls_source_model.transform(GemmToMatMul())
hls_source_model = qonnx.util.cleanup.cleanup_model(hls_source_model)
hls_source_model.save(str(hls_qonnx_path))

hls_config = hls4ml.utils.config_from_onnx_model(
    hls_source_model, granularity='name', backend='Vitis')
hls_model = hls4ml.converters.convert_from_onnx_model(
    hls_source_model,
    output_dir=str(ARTIFACT_DIR / 'hls4ml_jet_tagger'),
    backend='Vitis',
    hls_config=hls_config,
    part='xcu200-fsgd2104-2-e')
print('hls4ml project created; compilation and synthesis are optional follow-up steps')

We can draw the HLS model with hls4ml's built-in plotter. It shows the graph that hls4ml actually builds, annotated with the precision it assigned to the weights, biases, accumulators, and outputs of each layer. This view is distinct from the Netron view of the QONNX graph and from the PyTorch `draw_network` diagrams.

We check whether the weights now appear as narrow integer grids instead of hls4ml's default `fixed<16,6>`.

In [ ]:
hls4ml.utils.plot_model(
    hls_model,
    show_shapes=True,
    show_precision=True,
    to_file=None)

In the plot, the quantizers appear as integer grids. The activation quantizers are `ApplyAlpha` nodes with `fixed<4,4>` outputs at the inputs of the three hidden layers and `fixed<8,8>` at the input of the output layer. The `Dense` weights should follow the recipes as well: `fixed<3,3>` for the hidden layers and `fixed<8,8>` for the output layer.

The float scales do not disappear. hls4ml keeps them in separate `ApplyAlpha` multiplications, which is how the weights and activations stay on narrow integer grids.

### Takeaway

We converted the floating-point model to quantized Brevitas layers, calibrated its activation quantizers, recovered accuracy with Qronos and QAT, and exported the result to QONNX. The optional hls4ml step shows how that exported representation can become an HLS project for downstream hardware development; creating the project does not itself synthesize hardware or generate an FPGA bitstream.

This workflow uses standard quantizers and built-in conversion and export paths. The next section goes deeper by replacing quantizer components, controlling additional graph boundaries, learning bit widths, and defining custom export behavior.

# Custom quantizers and exploration

Research ideas and specialized deployment targets often require more control than predefined quantizers and layerwise replacement provide. 

This part of the tutorial shows how Brevitas can be extended while retaining its composable quantization model.

We will revisit the core concepts and experiment with:
- heavily customized quantizers, first inspecting the components of an integer quantizer and then replacing selected components;
- more flexible model conversion, using FX for functional calls and full-model replacement for explicit control over every boundary;
- constrained QAT, optimizing scale and bit-width parameters together with model weights; and
- a custom export flow, using handlers to map training-time quantizers to the required QONNX representation.



## Anatomy of an integer quantizer

An integer quantizer is assembled from several independent policies:

| Policy | Brevitas component name | Role | Typical reason to replace it |
| --- | --- | --- | --- |
| Statistics | `scaling_stats_impl` / `stats_impl` | Measures tensor range | Use max, percentile, MSE, or a custom statistic |
| Scale | `scaling_impl` | Maps the tensor range into the integer range | Use runtime, calibrated, or learned scales |
| Scale granularity | `scaling_per_output` / `scaling_shape` | Defines the scale shape | Choose per-tensor, per-channel, or per-group behavior |
| Zero point | `zero_point_impl` | Defines the affine offset | Select symmetric or asymmetric quantization |
| Bit width | `bit_width_impl` | Produces the integer precision | Keep it constant or make it trainable |
| Value rounding | `float_to_int_impl` | Maps scaled values to integers | Change forward rounding or its gradient estimator |
| Clamp | `tensor_clamp_impl` | Restricts the represented integer range | Match signedness, narrow range, or a target convention |
| Export | proxy handler | Defines the external representation | Map custom training behavior to export semantics |

The next cells inspect these pieces in a standard integer weight quantizer before replacing selected components.

### A predefined integer quantizer

We start with a standard Brevitas eight-bit per-output-channel weight quantizer. The module tree shows the concrete components created for one `QuantLinear` layer.

In [ ]:
from brevitas.quant import Int8WeightPerChannelFloat

standard_int_linear = qnn.QuantLinear(
    16, 8, weight_quant=Int8WeightPerChannelFloat, bias=False)
print('Quantizer recipe:', Int8WeightPerChannelFloat.__name__)

integer_quant = standard_int_linear.weight_quant.tensor_quant
modules_to_inspect = (
    'stats_impl', 'scaling_impl', 'zero_point_impl',
    'bit_width_impl', 'float_to_int_impl', 'tensor_clamp_impl')
print('Some of the components to inspect or override:')
for name, module in integer_quant.named_modules():
    if any(name.endswith(suffix) for suffix in modules_to_inspect):
        print(f'  {name}: {type(module).__name__}')

Within the quantizer, each one of these classes is composed of a series of nested subclasses as we can see below.

In [ ]:
print(standard_int_linear.weight_quant)

Given the nested modular structure, it might be tricky to modify high-level components that incorporates many subclasses.

Nevertheless, there are plenty of possibility to customise the quantizers to make them fit to your need, even by replacing entire hierarchy of classes.


For example, the rounding class maps a scaled value to a representable integer. `float_to_int_impl` controls this operation.

The scale path has more than one part. `stats_impl` computes statistics such as an absolute maximum from the tensor;\
`scaling_impl` turns those statistics into the external scale (e.g., combining it with the threshold);\
`bit_width_impl` supplies the integer precision, while `zero_point_impl` and `tensor_clamp_impl` determine the affine offset and represented range.


### Custom tensor-value rounding

Then we design our new class, keeping in mind the following:
- We DO NOT need to respect the init interface, we can add or remove arguments
- We DO need to respect the forward interface
- There might be other methods we need to carry over (e.g., state dict handling)

A good approach for this is to always inspect the original class to replace, and then try to understand what to carry over in the new class.




First we start by replacing the module used to round tensor values.
We implement RDFS (the Fourier-based gradient estimator for rounding) from StableQAT [4], replacing the default implementation of straight-through rounding.\
The idea is to have a function where the forward pass still uses `torch.round`, but the backward gradient has a smoother profile.


In [ ]:
import inspect
from IPython.display import Markdown, display
from brevitas.core.function_wrapper import RoundSte


def pretty_print_source(source):
    display(Markdown('```python\n' + source + '\n```'))


We first inspect the interface of the class we want to replace:

In [ ]:
source = inspect.getsource(RoundSte)  
pretty_print_source(source)

Then we create our new rounding class, and we visualise the difference in gradient behaviour:

In [ ]:
import math
from brevitas.core.function_wrapper import RoundSte
from notebook_utils import plot_rounding_gradients


# Define a new autograd function
class RoundRDFSFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        return torch.round(x)

    @staticmethod
    def backward(ctx, grad_output):
        x, = ctx.saved_tensors
        coefficient = 0.21 * math.sqrt(2.) * math.pi
        phase = (x + x.round()) * math.pi
        gradient_estimate = 2. / (1. + coefficient * torch.cos(phase)) - 1.
        return grad_output * gradient_estimate


# Interface around the new autograd function to avoid having to call `.apply`
class RoundRDFS(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return RoundRDFSFunction.apply(x)


plot_rounding_gradients({'RoundSte': RoundSte(), 'RoundRDFS': RoundRDFS()})

## Parameterized integer bit widths

Next we replace constant integer bit widths with learned parameters.

We are going to implement:
- one learned bit width per activation tensor
- one learned bit width per weight output channel

Brevitas already provides a scalar learned bit-width module. We define a per-channel implementation from scratch for weights, then attach it to an integer quantizer recipe. Both effective widths are restricted to integer values from 2 through 8.


### Custom activation quantizer

Activation quantization uses one bit-width parameter per tensor. The predefined integer recipe already knows how to instantiate Brevitas' scalar `BitWidthParameter`; we only select parameter-backed precision, its initial value, legal range, and RDFS value rounding.

In [ ]:
from dependencies import value
import brevitas.config as brevitas_config
from brevitas.core.bit_width import BitWidthImplType
from brevitas.core.function_wrapper import ScalarClampSte
from brevitas.core.restrict_val import IntRestrictValue
from brevitas.core.scaling import ScalingImplType
from brevitas.function import abs_binary_sign_grad
from brevitas.inject.enum import StatsOp
from brevitas.quant import Int8ActPerTensorFloat
from brevitas.quant import Int8WeightPerChannelFloat

In [ ]:
class LearnedIntAct(Int8ActPerTensorFloat):
    high_percentile_q = 99.0
    bit_width_impl_type = BitWidthImplType.PARAMETER
    restrict_bit_width_impl = IntRestrictValue
    bit_width = 8
    min_bit_width = 2
    bit_width_offset_min_val = 2
    bit_width_offset_max_val = 8
    float_to_int_impl = RoundRDFS


### Per-channel bit-width parameter module

We need to create our new per-channel bit-width parameter class.

As before, we first start by inspecting the class we need to replace.


In [ ]:
from brevitas.core.bit_width import BitWidthParameter
source = inspect.getsource(BitWidthParameter)  
pretty_print_source(source)


In this case, we add a new argument to the init function `bit_width_shape`, which will be resolved later through dependency injection.

We also need to preserve the state_dict dict method override; this allows us to load a floating-point checkpoint into a quantized models with no missing-keys error.

In [ ]:
class PerChannelBitWidthParameter(nn.Module):
    def __init__(
            self, bit_width, bit_width_shape,
            restrict_bit_width_impl,
            bit_width_offset_min_val, bit_width_offset_max_val,
            dtype=None, device=None):
        super().__init__()
        initial_value = torch.full(
            bit_width_shape, float(bit_width), dtype=dtype, device=device)
        self.bit_width_parameter = nn.Parameter(initial_value)
        self.min_bit_width = float(bit_width_offset_min_val)
        self.max_bit_width = float(bit_width_offset_max_val)
        self.restrict_bit_width = restrict_bit_width_impl
        self.clamp_bit_width = ScalarClampSte(
            self.min_bit_width, self.max_bit_width)

    def forward(self):
        bit_width = abs_binary_sign_grad(self.bit_width_parameter)
        bit_width = self.restrict_bit_width(bit_width)
        return self.clamp_bit_width(bit_width)

    def _load_from_state_dict(
            self, state_dict, prefix, local_metadata, strict, missing_keys,
            unexpected_keys, error_msgs):
        super()._load_from_state_dict(
            state_dict, prefix, local_metadata, strict, missing_keys,
            unexpected_keys, error_msgs)
        parameter_key = prefix + 'bit_width_parameter'
        if brevitas_config.IGNORE_MISSING_KEYS and parameter_key in missing_keys:
            missing_keys.remove(parameter_key)


### Custom weight quantizer

The weight recipe starts from the predefined per-channel integer quantizer. We replace its scalar constant bit width with the per-channel learned module and replace value rounding with RDFS, while leaving the existing proxy and integer-quantization core unchanged.

The shape arguments used below are injected from the parent weight module: `weight_ndims` is the rank of its weight tensor, `output_channel_dim` identifies the output-channel axis, and `out_channels` gives its size.

In [ ]:
class LearnedIntWeight(Int8WeightPerChannelFloat):
    @value
    def bit_width_shape(weight_ndims, output_channel_dim, out_channels):
        shape = [1] * weight_ndims
        shape[output_channel_dim] = out_channels
        return tuple(shape)

    scaling_impl_type = ScalingImplType.PARAMETER_FROM_STATS
    scaling_stats_op = StatsOp.MAX
    bit_width_impl_type = BitWidthImplType.PARAMETER
    restrict_bit_width_impl = IntRestrictValue
    bit_width_impl = PerChannelBitWidthParameter
    bit_width = 8
    bit_width_offset_min_val = 2
    bit_width_offset_max_val = 8
    float_to_int_impl = RoundRDFS

### Custom quantizer inspection

The activation recipe keeps one learned bit width per tensor, while the weight recipe keeps one per output channel. Both reuse the same RDFS rounding module.

In [ ]:
for quantizer in (LearnedIntWeight, LearnedIntAct):
    print(quantizer.__name__)
    print('  quantization: integer')
    print('  scale:', quantizer.scaling_impl_type)
    print('  bit width:', quantizer.bit_width_impl_type)
    print('  value rounding:', quantizer.float_to_int_impl)

The standard recipe gives us the complete layer/proxy/integer-quantization stack. The custom recipe keeps that stack and replaces selected policies:

| Component | Standard INT8 recipe | Learned integer recipe |
| --- | --- | --- |
| Scale | Predefined per-channel scale | Parameter-from-maximum-statistics per-channel scale |
| Bit width | Constant eight bits | Learned per output channel, bounded to 2–8 bits |
| Value rounding | Standard rounding | RDFS module |
| Export | Built-in handler | Custom handler for vector-valued bit-width metadata |

### Takeaway

An integer quantizer is a composition of policies for statistics, scaling, bit width, rounding, clamping, and export. The learned integer recipes reuse Brevitas's scaled-integer implementation and replace only the policies required for this experiment.

RDFS keeps ordinary rounded values in the forward pass while changing the backward gradient, and the same module can be shared by activation and weight recipes. The resulting recipes learn one bounded bit width per activation tensor and one per weight output channel. Next, we apply these recipes to complete models.

## FX-based graph replacement

Rather than defining a new model manually, we can apply the learned-integer recipes to the floating-point model through FX graph conversion. FX rewrites the functional ReLUs before inserting quantized modules and graph-level quantization boundaries.

In [ ]:
from brevitas.graph.quantize import preprocess_for_quantize
from brevitas.graph.quantize import quantize

learned_compute_layer_map = {
    nn.Linear: (
        qnn.QuantLinear,
        {'weight_quant': LearnedIntWeight, 'return_quant_tensor': True}),
}
learned_quant_act_map = {
    nn.ReLU: (
        qnn.QuantReLU,
        {'act_quant': LearnedIntAct, 'return_quant_tensor': True}),
}
learned_quant_identity_map = {
    role: (
        qnn.QuantIdentity,
        {'act_quant': LearnedIntAct, 'return_quant_tensor': True})
    for role in ('signed', 'unsigned')
}

fx_source_model = JetTagger().cpu()
fx_source_model.load_state_dict(fp32_state)
fx_preprocessed_model = preprocess_for_quantize(
    fx_source_model, trace_model=True)
fx_quant_model = quantize(
    fx_preprocessed_model,
    quant_identity_map=learned_quant_identity_map,
    compute_layer_map=learned_compute_layer_map,
    quant_act_map=learned_quant_act_map).to(DEVICE)
print('learned-integer FX graph:')
print(fx_quant_model.graph)

The converted graph exposes the ReLUs as modules and applies the learned-integer recipes at its input, activation, and linear boundaries. This model demonstrates graph-level conversion and remains isolated from the QAT path below.

## Full-model replacement

When exact tensor boundaries matter, we can define the learned-integer model explicitly. The dense module names match the FP32 model, so its checkpoint can initialize the new model while the quantizer state is created separately.

In [ ]:
from contextlib import contextmanager
import brevitas.config as brevitas_config


@contextmanager
def ignore_missing_quantizer_keys():
    previous_value = brevitas_config.IGNORE_MISSING_KEYS
    brevitas_config.IGNORE_MISSING_KEYS = True
    try:
        yield
    finally:
        brevitas_config.IGNORE_MISSING_KEYS = previous_value


class LearnedIntJetTagger(nn.Module):
    def __init__(self):
        super().__init__()
        self.input_quant = qnn.QuantIdentity(
            act_quant=LearnedIntAct, return_quant_tensor=True)
        self.fc1 = qnn.QuantLinear(16, 64, weight_quant=LearnedIntWeight)
        self.act1 = qnn.QuantReLU(
            act_quant=LearnedIntAct, return_quant_tensor=True)
        self.fc2 = qnn.QuantLinear(64, 32, weight_quant=LearnedIntWeight)
        self.act2 = qnn.QuantReLU(
            act_quant=LearnedIntAct, return_quant_tensor=True)
        self.fc3 = qnn.QuantLinear(32, 32, weight_quant=LearnedIntWeight)
        self.act3 = qnn.QuantReLU(
            act_quant=LearnedIntAct, return_quant_tensor=True)
        self.output = qnn.QuantLinear(32, 5, weight_quant=LearnedIntWeight)

    def forward(self, x):
        x = self.input_quant(x)
        x = self.act1(self.fc1(x))
        x = self.act2(self.fc2(x))
        x = self.act3(self.fc3(x))
        return self.output(x)

The detailed diagram reports effective integer bit widths. Activations use one width per tensor. Weights use one width per output channel, so the first three channels are shown for each linear layer.

In [ ]:
draw_network(
    LearnedIntJetTagger(),
    title='Learned-integer jet tagger',
    mode='detailed', orientation='horizontal')

### Takeaway

FX conversion can rewrite trace-visible operations such as functional ReLUs and insert quantization boundaries without redefining the model, but it depends on a traceable graph and boundaries that FX can observe. Full-model replacement requires more model-specific code, but makes every input, activation, and weight boundary explicit.

The FX model demonstrates automatic graph conversion. For the remaining exercises, we use the explicit model so that initialization, learned bit widths, QAT, and export all operate on known boundaries.

## Learned-bit-width QAT

We load the FP32 weights into the explicit learned-integer model, calibrate activation scales, and then optimize dense weights, scales, and bit-width parameters together.

Qronos is not used in this path because the representable integer ranges can change during QAT. It could be tested after bit widths are frozen, but that combination is outside this tutorial.

### Reading learned bit widths

Before QAT, we need helpers that read the effective integer bit width from every weight and activation quantizer. We also define `bit_width_cost`, which averages those widths.

In [ ]:
def learned_bit_widths(model, force_eval=False):
    bit_widths = {}
    for name, module in model.named_modules():
        if isinstance(module, qnn.QuantLinear):
            weight = module.quant_weight()
            bit_widths[f'{name}.weight'] = weight.bit_width
        elif isinstance(module, (qnn.QuantIdentity, qnn.QuantReLU)):
            bit_widths[f'{name}.activation'] = (
                module.act_quant.bit_width(force_eval=force_eval))
    return bit_widths


def bit_width_cost(model):
    totals = [bit_width.float().mean()
              for bit_width in learned_bit_widths(model).values()]
    return torch.stack(totals).mean()

### Checking learned bit widths

The stored parameters are continuous, while the forward pass applies straight-through integer rounding and clamps effective values to 2–8 bits. The checker below verifies those effective widths before training and export.

In [ ]:
def check_bit_widths(model):
    checks = []
    bit_widths = learned_bit_widths(model, force_eval=True)
    if not bit_widths:
        raise RuntimeError('No learned bit widths were found')
    for name, bit_width in bit_widths.items():
        bit_width = bit_width.detach()
        module_name, role = name.rsplit('.', 1)
        expected_values = (
            model.get_submodule(module_name).out_features
            if role == 'weight' else 1)
        valid = (
            torch.isfinite(bit_width)
            & (2. <= bit_width) & (bit_width <= 8.)
            & (bit_width == bit_width.round())).all().item()
        valid = valid and bit_width.numel() == expected_values
        values = torch.unique(bit_width.flatten())
        checks.append(valid)
        print(
            f'{name}: shape={tuple(bit_width.shape)} '
            f'bit_widths={values.cpu().tolist()} valid={valid}')
    return all(checks)

### Initializing the learned-integer model

We load the FP32 weights into the explicit model and calibrate the activation scales before QAT.

In [ ]:
qat_model = LearnedIntJetTagger()
with ignore_missing_quantizer_keys():
    qat_model.load_state_dict(fp32_model.state_dict())
qat_model.eval().to(DEVICE)
if not check_bit_widths(qat_model):
    raise RuntimeError('Invalid initial integer bit width')

with torch.no_grad(), calibration_mode(qat_model):
    for features, _ in calibration_loader:
        qat_model(features.to(DEVICE))

evaluate(qat_model)

Calibration has initialized the activation quantizer state from representative data.

We can now start QAT. Gradients reach dense weights, scales, and bit-width parameters.

A continuous bit-width parameter can change without crossing an integer rounding boundary, so its effective width may remain unchanged during this short run.

In [ ]:
bit_width_parameters_before = {
    name: parameter.detach().cpu().clone()
    for name, parameter in qat_model.named_parameters()
    if 'bit_width' in name}
if not bit_width_parameters_before:
    raise RuntimeError('No trainable bit-width parameters were found')

optimizer = torch.optim.Adam(qat_model.parameters(), lr=1e-1)
losses = train_steps(
    qat_model, train_loader, optimizer, max_steps=80,
    regularizer=lambda model: 1e-0 * bit_width_cost(model))


The QAT steps have completed. We now inspect the effective integer bit widths produced by the short training run.

In [ ]:
draw_network(
    qat_model,
    title='Learned-integer jet tagger',
    mode='detailed', orientation='horizontal')

### Takeaway

The learned-integer model starts from the FP32 dense weights, while representative data initializes its activation scales. During QAT, gradients update dense weights, scales, and continuous bit-width parameters together; rounding and clamping determine the effective integer widths used in the forward pass.

Before export, effective widths must be checked for integer values, the allowed 2–8-bit range, and the expected scalar or per-channel shape. The next section defines custom handlers that preserve this heterogeneous metadata in QONNX.

## Custom QONNX export

The learned-integer model uses a notebook-local rounding component and per-channel bit-width metadata. We therefore need custom handlers that define how these values are represented in QONNX.

RDFS changes only the backward pass, so its inference representation is `ROUND`. A component with different forward semantics would also require a new symbolic and backend implementation.

### Exporting learned integer metadata

A QONNX `Quant` node carries scale, zero point, bit width, signedness, narrow-range state, and rounding mode. This helper assembles those fields and maps RDFS to `ROUND` because only its backward pass differs.

In [ ]:
from brevitas.export.onnx.qonnx.function import BrevitasQuantOp
from brevitas.export.onnx.qonnx.handler import BrevitasActQuantProxyHandler
from brevitas.export.onnx.qonnx.handler import BrevitasWeightQuantProxyHandler
from brevitas.export.onnx.qonnx.manager import QONNXManager


def rdfs_integer_export_metadata(
        scale, zero_point, bit_width, narrow_range, signed):
    return {
        'scale': scale,
        'zero_point': zero_point,
        'bit_width': bit_width,
        'narrow_range': narrow_range,
        'signed': signed,
        'rounding_mode': 'ROUND'}

### Weight and activation handlers

Next we define separate integer handlers for weights and activations. The weight handler reads vector-valued bit widths from the quantized weight without applying scalar binary checks. The activation handler reads its scalar width from the activation proxy. Both verify that the proxy uses `RoundRDFS`.

In [ ]:
def require_rdfs_rounding(module):
    if hasattr(module, 'fused_activation_quant_proxy'):
        rounding_impl = module.fused_activation_quant_proxy.tensor_quant.int_quant.float_to_int_impl
    else:
        rounding_impl = module.tensor_quant.int_quant.float_to_int_impl
    if not isinstance(rounding_impl, RoundRDFS):
        raise RuntimeError('TutorialQONNXManager only handles RoundRDFS integer proxies')


class RDFSWeightQuantProxyHandler(BrevitasWeightQuantProxyHandler):
    def prepare_for_export(self, module):
        if module.is_quant_enabled:
            require_rdfs_rounding(module)
            quant_weight = module.tracked_module_list[0].quant_weight()
            self.quant_weight_values = {
                tracked.weight.data_ptr(): tracked.quant_weight().value
                for tracked in module.tracked_module_list}
            self.symbolic_kwargs = rdfs_integer_export_metadata(
                quant_weight.scale, quant_weight.zero_point,
                quant_weight.bit_width, module.is_narrow_range,
                quant_weight.signed)
            self.quant_op = BrevitasQuantOp


class RDFSActQuantProxyHandler(BrevitasActQuantProxyHandler):
    def prepare_for_export(self, module):
        if module.is_quant_enabled:
            require_rdfs_rounding(module)
            self.validate(module)
            self.symbolic_kwargs = rdfs_integer_export_metadata(
                module.scale(), module.zero_point(), module.bit_width(),
                module.is_narrow_range, module.is_signed)
            self.quant_op = BrevitasQuantOp




### Registering the handlers

With the handlers defined, we register them before the standard QONNX handlers. This ensures that they are selected for the integer proxies used by the tutorial model.

In [ ]:
class TutorialQONNXManager(QONNXManager):
    # Handler lookup uses the first exact proxy-type match.
    handlers = [
        RDFSWeightQuantProxyHandler,
        RDFSActQuantProxyHandler,
        *QONNXManager.handlers]


print('Custom export mappings:')
print('  RoundRDFS weight forward -> QONNX Quant[ROUND]')
print('  RoundRDFS activation forward -> QONNX Quant[ROUND]')

### Exporting the learned model

The export manager is now ready. We check the learned bit widths once more, export the model, and load the resulting ONNX graph.

In [ ]:
import onnx
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.transformation.infer_shapes import InferShapes

custom_export_device = next(qat_model.parameters()).device
custom_export_training = qat_model.training
qat_model.cpu().eval()
qonnx_path = ARTIFACT_DIR / 'jet_tagger_learned_integer_qonnx.onnx'
example_input = torch.from_numpy(X_test[:1])
try:
    if not check_bit_widths(qat_model):
        raise RuntimeError('Refusing to export an invalid learned bit width')
    TutorialQONNXManager.export(
        qat_model, args=example_input, export_path=str(qonnx_path),
        opset_version=14)
finally:
    qat_model.to(custom_export_device)
    qat_model.train(custom_export_training)

onnx_model = onnx.load(qonnx_path)
onnx.checker.check_model(onnx_model)
qonnx_model = ModelWrapper(onnx_model).transform(InferShapes())

After a successful export and `onnx.checker` validation, the following cell opens the serialized QONNX graph in Netron. Selecting a `Quant` node shows its bit-width input: activation quantizers carry one width, and weight quantizers carry one width per output channel. This inference graph is separate from the PyTorch model structure drawn earlier.

In [ ]:
from notebook_utils import show_netron

show_netron(qonnx_path)

If export and validation complete successfully, we can redraw the PyTorch model and read the bit-width initializer of each QONNX `Quant` node. This lets us compare the model's per-channel widths with the serialized metadata.

In [ ]:
draw_network(
    qat_model,
    title='Exported learned-integer bit widths',
    mode='detailed',
    orientation='horizontal')

print('QONNX Quant bit-width initializers:')
for node in qonnx_model.graph.node:
    if node.op_type != 'Quant':
        continue
    bit_width = qonnx_model.get_initializer(node.input[3])
    if bit_width is not None:
        flat = bit_width.reshape(-1)
        print(f'  {node.name}: shape={bit_width.shape}, first={flat[:3].tolist()}')

The intended metadata read-back distinguishes `Quant` nodes with scalar activation widths from those with vector-valued per-channel weight widths. This exercise stops at graph validation because support for vector-valued `Quant` bit-width metadata depends on the downstream runtime or hardware backend.

### Takeaway

Custom proxy handlers translate the training-time quantizers into deployment metadata. RDFS maps to ordinary `ROUND` at inference because only its backward behavior differs, while the weight handler preserves vector-valued per-channel bit widths.

ONNX checking and metadata read-back validate graph structure and serialization; they do not establish that a runtime or hardware backend supports vector-valued `Quant` metadata. Unlike the fixed-integer path, which includes a runtime parity check, this learned-width path stops at graph validation and inspection. A quantizer with new forward semantics would additionally require an operator schema, shape inference, reference execution, and backend lowering.

## Ideas for further experiments

The cells in the second part can be edited and rerun. Some useful starting points are:

### Quantizer recipe

- Keep bit widths fixed before making them trainable.
- Try a predefined MSE-based scale implementation.
- Change scale granularity or the scale statistics.
- Compare per-tensor and per-channel learned bit widths.

### Format parameters

- Change the initial integer bit width.
- Change the minimum and maximum bit-width bounds.
- Change the per-channel initialization pattern.
- Compare stored continuous parameters with effective rounded bit widths.

### Rounding

- Replace RDFS with the standard rounding module, or with a different function for forward/backward.

### Model conversion

- Change the layerwise callback used in the first section.
- Modify the FX activation or identity maps.
- Add or remove explicit activation boundaries in the full-model definition.
- Change which layers return `QuantTensor`.

### QAT and export

- Change the number of QAT steps, learning rate, or bit-width-cost coefficient.
- Inspect whether stored bit-width parameters move without changing the rounded width.
- Change the per-channel bit-width pattern before export.
- Inspect the resulting bit-width initializer shapes in the ONNX graph.

When an experiment changes a class or model definition, rerun that definition and all dependent cells below it. Recreate objects instead of trying to undo their state in place.

# Summary

We have followed two Brevitas workflows: a fixed-format path and a specialized integer path with learned precision and custom export semantics.

| Requirement | Recommended path |
| --- | --- |
| Standard model quantization | Layerwise conversion |
| Fixed-format PTQ | Calibration and Qronos |
| Standard QAT | Qronos-refined integer model |
| Standard export | Built-in QONNX/QCDQ |
| Functional graph rewriting | FX |
| Explicit boundary control | Full-model replacement |
| Custom integer precision and gradients | Quantizer components |
| Learned heterogeneous integer precision | Constrained QAT |
| Custom deployment semantics | Export handlers |

# References

1. [Brevitas tutorials](https://xilinx.github.io/brevitas/master/tutorials/index.html)
2. [TVMCon 2021 Brevitas tutorial](https://xilinx.github.io/brevitas/master/tutorials/tvmcon2021.html)
3. [hls4ml Brevitas jet-tagger tutorial](https://github.com/fastmachinelearning/hls4ml-tutorial/blob/main/2_quantization/2b_brevitas.ipynb)
4. [Microsoft StableQAT](https://github.com/microsoft/StableQAT)
5. [Qronos](https://openreview.net/forum?id=7axclBCYul)
6. [QONNX](https://github.com/fastmachinelearning/qonnx)
7. [OpenML dataset 42468](https://www.openml.org/d/42468)
8. [Duarte et al., 2018](https://arxiv.org/abs/1804.06913)